In [21]:
%run lib.py
%run init.ipynb

33 datasets found
Setup Complete
Datasets Found: 33
agri_large
agri_medium
agri_small
energy_large
energy_medium
energy_small
favorita-train_large
favorita-train_medium
favorita-train_small
favorita-transaction_large
favorita-transaction_medium
favorita-transaction_small
m4-daily_large
m4-daily_medium
m4-daily_small
m4-hourly_large
m4-hourly_medium
m4-hourly_small
m4-monthly_large
m4-monthly_medium
m4-monthly_small
m4-quarterly_large
m4-quarterly_medium
m4-quarterly_small
m4-weekly_large
m4-weekly_medium
m4-weekly_small
m4-yearly_large
m4-yearly_medium
m4-yearly_small
traffic_large
traffic_medium
traffic_small


In [22]:
from statsmodels.tsa.arima.model import ARIMA

from statsmodels.tsa.statespace.sarimax import SARIMAX

from tqdm.auto import tqdm

In [23]:
ARIMA_ORDER = (
    1,
    1,
    1
)

SARIMA_ORDER = (
    1,
    1,
    1
)

SARIMA_SEASONAL_ORDER = (
    1,
    1,
    1,
    12
)


In [24]:
def naive_forecast(
    train,
    horizon
):

    return np.repeat(
        train[-1],
        horizon
    )

In [25]:
def arima_forecast(
    train,
    horizon
):

    try:

        model = ARIMA(
            train,
            order=ARIMA_ORDER
        )

        fitted = model.fit()

        forecast = fitted.forecast(
            steps=horizon
        )

        return np.asarray(
            forecast
        )

    except:

        return naive_forecast(
            train,
            horizon
        )

In [26]:
def sarima_forecast(
    train,
    horizon
):

    try:

        model = SARIMAX(
            train,
            order=SARIMA_ORDER,
            seasonal_order=SARIMA_SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fitted = model.fit(
            disp=False
        )

        forecast = fitted.forecast(
            steps=horizon
        )

        return np.asarray(
            forecast
        )

    except:

        return naive_forecast(
            train,
            horizon
        )

In [27]:
def evaluate_dataset(
    df,
    forecast_function,
    horizon
):

    y_true_all = []

    y_pred_all = []

    for _, group in tqdm(
        df.groupby("unique_id"),
        leave=False
    ):

        group = group.sort_values(
            "ds"
        )

        if len(group) <= horizon:
            continue

        train = (
            group["y"]
            .iloc[:-horizon]
            .values
        )

        test = (
            group["y"]
            .iloc[-horizon:]
            .values
        )

        pred = forecast_function(
            train,
            horizon
        )

        pred = pred[:horizon]

        y_true_all.extend(
            test
        )

        y_pred_all.extend(
            pred
        )

    metrics = calculate_metrics(
        y_true_all,
        y_pred_all
    )

    return metrics

In [ ]:
results = []

MODEL_MAP = {

    "Naive":
        naive_forecast,

    "ARIMA":
        arima_forecast,

    "SARIMA":
        sarima_forecast
}

for file in CSV_FILES:

    dataset_name = file.stem

    print()

    print("=" * 80)

    print(dataset_name)

    horizon = get_horizon(
        dataset_name
    )

    df = load_dataset(
        file
    )

    summary = dataset_summary(
        df
    )

    print(summary)

    for model_name, model_func in MODEL_MAP.items():

        print(
            f"Running {model_name}"
        )

        start = time()

        metrics = evaluate_dataset(
            df,
            model_func,
            horizon
        )

        runtime = time() - start

        result = build_result_row(
            dataset_name=dataset_name,
            model_name=model_name,
            horizon=horizon,
            runtime=runtime,
            metrics=metrics,
            df=df
        )

        results.append(
            result
        )


agri_large
{'n_rows': 15343, 'n_series': 246, 'start': Timestamp('1970-01-01 00:00:00.000001961'), 'end': Timestamp('1970-01-01 00:00:00.000002024')}
Running Naive


  0%|          | 0/246 [00:00<?, ?it/s]

Running ARIMA


  0%|          | 0/246 [00:00<?, ?it/s]

Running SARIMA


...

  0%|          | 0/370 [00:00<?, ?it/s]

Running ARIMA


  0%|          | 0/370 [00:00<?, ?it/s]

Running SARIMA


  0%|          | 0/370 [00:00<?, ?it/s]

In [ ]:
results_df = pd.DataFrame(
    results
)

results_df.head()

In [ ]:
results_df.to_csv(
    RESULTS_DIR /
    "statistical_models_results.csv",
    index=False
)

In [ ]:
results_df.sort_values(
    [
        "dataset",
        "MAE"
    ]
)

In [ ]:
results_df.groupby(
    "model"
)[
    [
        "MAE",
        "RMSE",
        "SMAPE"
    ]
].mean().sort_values(
    "MAE"
)

In [ ]:
best = (

    results_df

    .sort_values(
        "MAE"
    )

    .groupby(
        "dataset"
    )

    .first()
)

best["model"].value_counts()